**Ingest Drivers.csv file**
1. Read file using spark dataframe reader API
2. Define and enforce schema (preserve the nested structure)
3. add Metadata Columns 
-   source file
- ingestion Timestamp
3. write to bonze delta table

In [0]:
%run ../00.Common/01.environment-config


In [0]:
%run ../00.Common/02.bronze-helpers


In [0]:
source_file = f"{landing_folder_path}/drivers.json"
table_name = f"{catalog_nameone}.{bronze_schema}.drivers"

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DateType

name_schema = StructType([
    StructField("givenName", StringType()),
    StructField("familyName", StringType()),
])

drivers_schema = StructType([
    StructField("driverId", StringType()),
    StructField("name", name_schema),
    StructField("dateOfBirth", DateType()),
    StructField("nationality", StringType()),
    StructField("url", StringType()),

])

drivers_df = (
    spark.read.format("json")
    .schema(drivers_schema)
    .option("mode", "FAILFAST")
    .load(source_file)
)

drivers_final_df = add_ingestion_metadata(drivers_df)
display(drivers_final_df)

In [0]:
drivers_final_df.write.format('delta').mode('overwrite').saveAsTable(table_name)